# 03.3 Attention Intro

`Attention` is one of the core concepts of the Transformer era.

If we summarize its intuition in one sentence:

- when processing one position, the model dynamically decides which other positions in the sequence to attend to

This notebook does not aim to cover the full Transformer yet; it aims to make the minimal attention mechanism understandable.


## Learning Goals

After this notebook, you should be able to:

1. Understand the basic roles of `query
2. Manually compute a small attention example.
3. Understand attention scores, softmax weights, and context vectors.
4. Understand why we divide by `sqrt(d_k)`.
5. Understand the role of masking in attention.
6. Write a minimal self-attention module in `PyTorch`.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

## A Minimal Attention Example

The core attention computation can be summarized as:

1. `scores = QK^T`
2. `weights = softmax(scores)`
3. `context = weights * V`

We start with very small matrices here.


In [ ]:
Q = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
K = torch.tensor([[1.0, 0.0], [0.5, 1.0]])
V = torch.tensor([[10.0, 0.0], [0.0, 20.0]])

scores = Q @ K.T
weights = F.softmax(scores, dim=-1)
context = weights @ V

print("Q =\n", Q)
print("K =\n", K)
print("V =\n", V)
print("scores =\n", scores)
print("weights =\n", weights)
print("context =\n", context)

Intuition:

- what I am looking for right now
- what features each position has
- the content that will actually be mixed together

Note: this is only the minimal intuition version, not the only valid interpretation.


## Why Divide by `sqrt(d_k)`?

When the dimension `d_k` grows, dot-product values also tend to grow.

If the scores become too large, the `softmax` can become overly sharp.

So the common formula is:

- `scores = QK^T

In [ ]:
Q = torch.randn(2, 8)
K = torch.randn(2, 8)

raw_scores = Q @ K.T
scaled_scores = raw_scores / math.sqrt(Q.size(-1))

print("raw_scores =\n", raw_scores)
print("scaled_scores =\n", scaled_scores)
print("softmax(raw_scores) =\n", F.softmax(raw_scores, dim=-1))
print("softmax(scaled_scores) =\n", F.softmax(scaled_scores, dim=-1))

## 3. Self-Attention

`Self-Attention` means:

- `Q`, `K`, and `V` all come from the same input sequence

This is the core case used inside Transformers.


In [ ]:
x = torch.tensor(
    [
        [1.0, 0.0, 1.0],
        [0.0, 1.0, 1.0],
        [1.0, 1.0, 0.0],
    ]
)

Q = x
K = x
V = x

scores = (Q @ K.T) / math.sqrt(x.size(-1))
weights = F.softmax(scores, dim=-1)
context = weights @ V

print("x =\n", x)
print("scores =\n", scores)
print("weights =\n", weights)
print("context =\n", context)

Here every position can "look at" other positions in the sequence.

This is very different from the step-by-step propagation style of an `LSTM`.


## 4. Padding Mask

If some positions are only padding, the model should not attend to them.

A common approach is to set the scores at those positions to a very large negative value.


In [ ]:
scores = torch.tensor([[2.0, 1.0, 0.5]])
padding_mask = torch.tensor([[False, False, True]])

masked_scores = scores.masked_fill(padding_mask, float("-inf"))
weights = F.softmax(masked_scores, dim=-1)

print("scores =", scores)
print("masked_scores =", masked_scores)
print("weights =", weights)

This makes the attention weight of the final padding position become 0.


## 5. Causal Mask

In autoregressive settings, a position should not see future positions.

In that case, we use a causal mask.


In [ ]:
seq_len = 4
causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)

scores = torch.randn(seq_len, seq_len)
masked_scores = scores.masked_fill(causal_mask, float("-inf"))
weights = F.softmax(masked_scores, dim=-1)

print("causal_mask =\n", causal_mask)
print("weights =\n", weights)

From this weight matrix, you can see that each row can only attend to itself and the positions before it.


## A Minimal Self-Attention Module

Below we implement a very small single-head self-attention module.


In [ ]:
class SimpleSelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(Q.size(-1))

        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        context = weights @ V
        return context, weights


attn = SimpleSelfAttention(d_model=4)
x = torch.randn(2, 5, 4)
context, weights = attn(x)

print("x.shape =", x.shape)
print("context.shape =", context.shape)
print("weights.shape =", weights.shape)

The shapes here are very important:

- `x.shape == (batch_size, seq_len, d_model)`
- `weights.shape == (batch_size, seq_len, seq_len)`
- `context.shape == (batch_size, seq_len, d_model)`

That means each position assigns one set of attention weights over the whole sequence.


In [ ]:
# Exercise 1
# Given x.shape == (3, 7, 16)
# If you use single-head self-attention, what are context.shape and weights.shape?

Exercise 1 Reference Answer

For input `x.shape == (3, 7, 16)`, single-head self-attention returns `context.shape == (3, 7, 16)` and `weights.shape == (3, 7, 7)`.

In [ ]:
# Exercise 2
# Given a scores vector [2.0, 1.0, -1.0],
# compute the attention weights with softmax.

# scores =
# weights =
# print(weights)

In [ ]:
# Exercise 2 Reference Solution

scores = torch.tensor([2.0, 1.0, -1.0])
weights = F.softmax(scores, dim=-1)
print(weights)

## One Intuitive Contrast with LSTM

For now, keep one practical difference in mind:

- propagates information step by step across time
- one position can directly look at all positions

This is not a complete comparison, but it is enough to build an initial intuition.


## Summary

The most important outcome of this notebook is understanding the three-step computation of attention:

1. dot-product scoring
2. turn scores into weights with `softmax`
3. take a weighted sum over `V`

You should now be able to answer:

1. What roles do `Q`, `K`, and `V` play?
2. Why do we use `softmax` in attention?
3. Why do we divide by `sqrt(d_k)`?
4. What problems do padding masks and causal masks solve?

Suggested next step:

- If you continue, the next natural step is a `Transformer Encoder` notebook that places attention into a fuller structure.